In [333]:
import pandas as pd
import numpy as np

## Table 2 Creation

Continuing with Table 2, as specified in the assignment guidelines, the relevant data are provided in the files 
<span style="background-color: #f2f2f2; padding:2px 6px; border-radius:4px;">2_20241226_ecoData.csv</span> 
and 
<span style="background-color: #f2f2f2; padding:2px 6px; border-radius:4px;">2_20250108_doughnutv3-data.csv</span>.

It should be noted that the file 
<span style="background-color: #f2f2f2; padding:2px 6px; border-radius:4px;">2_20250108_doughnutv3-data.csv</span> 
contains both social and ecological indicators. Therefore, prior to analysis, the dataset must be filtered based on the values of the 
<span style="background-color: #f2f2f2; padding:2px 6px; border-radius:4px;">domain</span> 
column in order to isolate the relevant subset of indicators.

In [334]:
eco1 = pd.read_csv(r'a-fanning-doughnut-v3-a0460e5\Analysis-Final\myData\2_20241226_ecoData.csv')
socio_eco = pd.read_csv(r'a-fanning-doughnut-v3-a0460e5\Analysis-Final\myData\2_20250108_doughnutv3-data.csv')

In [335]:
eco2 = socio_eco[socio_eco['domain'] == 'ecological']

In [336]:
eco = pd.concat([eco1, eco2])

In [337]:
eco.head()

,domain,dimension,type,group,indicator,date,populationTotal,GNIperCap,value
0,ecological,climate change,global doughnut,World,co2_ppm,2000,NaN,NaN,368.96
1,ecological,climate change,global doughnut,World,co2_ppm,2001,NaN,NaN,370.57
2,ecological,climate change,global doughnut,World,co2_ppm,2002,NaN,NaN,372.58
3,ecological,climate change,global doughnut,World,co2_ppm,2003,NaN,NaN,375.14
4,ecological,climate change,global doughnut,World,co2_ppm,2004,NaN,NaN,376.95


The dataset requires additional preprocessing, as it includes indicators reported at both global and national levels.

To ensure consistency in the analysis, the data should be filtered based on the <span style="background-color: #f2f2f2; padding:2px 6px; border-radius:4px;">type</span> column, retaining only global-level observations.

Filtering based on the <span style="background-color: #f2f2f2; padding:2px 6px; border-radius:4px;">group</span> column is unnecessary. In this dataset, entries where <span style="background-color: #f2f2f2; padding:2px 6px; border-radius:4px;">type = "global doughnut"</span> correspond exclusively to <span style="background-color: #f2f2f2; padding:2px 6px; border-radius:4px;">group = "World"</span>. Therefore, filtering by <span style="background-color: #f2f2f2; padding:2px 6px; border-radius:4px;">type</span> alone is sufficient to isolate the desired observations.

In [338]:
eco = eco[eco['type'] != 'national aggregate']

In [339]:
# drop unnecessary columns
eco = eco.drop(columns=['type', 'group', 'populationTotal', 'GNIperCap', 'domain'])

In [340]:
# see state of missing values
eco.isna().sum()

dimension     0
indicator     0
date          0
value        70
dtype: int64

Rows with missing values in the indicator column should be removed, as this variable is essential for interpretation and cannot be reliably inferred or imputed.

In [341]:
eco = eco[eco['value'].notna()]

In [342]:
eco_ind = (
    eco.sort_values('date')
       .groupby(['dimension', 'indicator'])
       .agg({
           'date': ['first', 'last'],
           'value': ['first', 'last']
       })
)

In [343]:
eco_ind = eco_ind.round(2)
eco_ind = eco_ind.reset_index()
eco_ind

dimension          indicator  date          value         
                                              first  last    first     last
0            air pollution        interhemAOD  2000  2022     0.08     0.08
1   biodiversity breakdown     extinction1900  2000  2022   100.00   100.00
2   biodiversity breakdown           hanppGtC  2000  2020    14.60    16.80
3       chemical pollution  EUshare_hzdHealth  2000  2022    76.59    76.59
4       chemical pollution        chemicalsMt  2000  2017  1186.00  2276.00
5           climate change            co2_ppm  2000  2022   368.96   417.07
6           climate change            erf_wm2  2000  2022     1.77     2.91
7    freshwater disruption            blueDev  2000  2005     0.18     0.19
8    freshwater disruption            soilDev  2000  2014    15.43    19.25
9          land conversion     forestAreaMKM2  2000  2020    39.21    38.27
10      nutrient pollution         nitrogenMt  2000  2022   132.24   194.34
11      nutrient pollution       phosphorusMt  2000  2022    13.55    23.17
12     ocean acidification            omega_a  2000  2022     2.99     2.79
13         ozone depletion         totalOzone  2000  2021   284.30   284.58

As observed in the table above (line 3), the first chemical pollution indicator refers specifically to the share of EU chemical pollution (<span style="background-color: #f2f2f2; padding:2px 6px; border-radius:4px;">EUshare_hzdHealth</span>). This does not align with the global scope required for the analysis. Therefore, this row should be removed from the dataset.

In [344]:
table_2 = eco_ind[eco_ind['indicator'] != 'EUshare_hzdHealth'].copy()
table_2 = table_2.round(2) # use two decimals like in assignment sheet
table_2

dimension       indicator  date          value         
                                           first  last    first     last
0            air pollution     interhemAOD  2000  2022     0.08     0.08
1   biodiversity breakdown  extinction1900  2000  2022   100.00   100.00
2   biodiversity breakdown        hanppGtC  2000  2020    14.60    16.80
4       chemical pollution     chemicalsMt  2000  2017  1186.00  2276.00
5           climate change         co2_ppm  2000  2022   368.96   417.07
6           climate change         erf_wm2  2000  2022     1.77     2.91
7    freshwater disruption         blueDev  2000  2005     0.18     0.19
8    freshwater disruption         soilDev  2000  2014    15.43    19.25
9          land conversion  forestAreaMKM2  2000  2020    39.21    38.27
10      nutrient pollution      nitrogenMt  2000  2022   132.24   194.34
11      nutrient pollution    phosphorusMt  2000  2022    13.55    23.17
12     ocean acidification         omega_a  2000  2022     2.99     2.79
13         ozone depletion      totalOzone  2000  2021   284.30   284.58

Lastly, to ensure consistency with the assignment example, the values in the <span style="background-color: #f2f2f2; padding:2px 6px; border-radius:4px;">indicator</span> column must be replaced with the appropriate descriptive names of the corresponding indicators used in the analysis.

Since the data documents do not include indicator descriptions, the descriptions presented in the assignment examples will be copied to reach the requested form for table 2.

In [345]:
indicator_descr = {
    "interhemAOD": "Arithmetic error asymmetry between Earth's hemispheres of sunlight reaching the surface, owing to differences in atmospheric particle concentration (at most 0.1 inter-hemispheric difference in Aerosol Optical Depth).",
    "extinction1900": "Rate of species extinctions per million species years (at most 10 E/MSY).",
    "hanppGtC": "Human appropriation of net primary productivity, billions of tonnes of carbon per year (at most 10% of 55.9 Gt C).",
    "chemicalsMt": "Production of hazardous chemicals, millions of tonnes per year (at most 5% of the 1,200 Mt of total chemicals produced in year 2000).",
    "co2_ppm": "Atmospheric carbon dioxide concentration, parts per million (at most 350 ppm CO2).",
    "erf_wm2": "Human-induced radiative forcing at the top of the atmosphere, Watt per square metre (at most 1 W m**(-2)).",
    "blueDev": "Proportion of land area with human-induced disturbance of blue-water flow deviating from Holocene variability (at most 10.2%).",
    "soilDev": "Proportion of land area with root-zone soil moisture deviating from Holocene variability (at most 11.1%).",
    "forestAreaMKM2": "Area of forested land as a proportion of forest-covered land before human alteration (at least 75% of 64 million square kilometres).",
    "nitrogenMt": "Nitrogen applied to land as fertilizer, millions of tonnes per year (at most 62 Mt per year).",
    "phosphorusMt": "Phosphorus applied to land as fertilizer, millions of tonnes per year (at most 6.2 Mt per year).",
    "omega_a": "Average saturation state of aragonite at the ocean surface (at least 80% of pre-industrial saturation state of 3.44 Ωarag).",
    "totalOzone": "Concentration of ozone in the stratosphere, Dobson units (at most 5% decrease with respect to the 1964–1980 value of 290 DU)."
}

In [346]:
table_2['indicator'] = table_2['indicator'].map(indicator_descr)

In [347]:
table_2

dimension                                          indicator  \
                                                                                
0            air pollution  Arithmetic error asymmetry between Earth's hem...   
1   biodiversity breakdown  Rate of species extinctions per million specie...   
2   biodiversity breakdown  Human appropriation of net primary productivit...   
4       chemical pollution  Production of hazardous chemicals, millions of...   
5           climate change  Atmospheric carbon dioxide concentration, part...   
6           climate change  Human-induced radiative forcing at the top of ...   
7    freshwater disruption  Proportion of land area with human-induced dis...   
8    freshwater disruption  Proportion of land area with root-zone soil mo...   
9          land conversion  Area of forested land as a proportion of fores...   
10      nutrient pollution  Nitrogen applied to land as fertilizer, millio...   
11      nutrient pollution  Phosphorus applied to land as fertilizer, mill...   
12     ocean acidification  Average saturation state of aragonite at the o...   
13         ozone depletion  Concentration of ozone in the stratosphere, Do...   

    date          value           
   first  last    first     last  
0   2000  2022     0.08     0.08  
1   2000  2022   100.00   100.00  
2   2000  2020    14.60    16.80  
4   2000  2017  1186.00  2276.00  
5   2000  2022   368.96   417.07  
6   2000  2022     1.77     2.91  
7   2000  2005     0.18     0.19  
8   2000  2014    15.43    19.25  
9   2000  2020    39.21    38.27  
10  2000  2022   132.24   194.34  
11  2000  2022    13.55    23.17  
12  2000  2022     2.99     2.79  
13  2000  2021   284.30   284.58